<a href="https://colab.research.google.com/github/fatmayddn/pandas-data-cleaning/blob/main/pandas_veri_temizleme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pandas ile Veri Temizleme
**İçerik:**
1. Veriyi Yükleme ve İlk Bakış
2. Genel Veri Sağlığı Kontrolü
3. Eksik Değerlerin Tespiti ve Giderilmesi
4. Yinelenen (Duplicate) Kayıt Kontrolü
5. Veri Tiplerinin Düzeltilmesi (Tarih Sütunu)
6. Mantıksal Hata / Aykırı Değer Tespiti ve Düzeltilmesi
7. Kategori Tutarlılığı Kontrolü
8. İndeks Düzenleme
9. Temizlenmiş Veriyi Kaydetme
10. Uçtan Uca Temizleme Fonksiyonu

## 1. Veriyi Yükleme ve İlk Bakış

In [1]:
import pandas as pd
import numpy as np

df_ham = pd.read_csv('/content/retail_store_sales.csv')
print("Boyut:", df_ham.shape)
df_ham.head()

Boyut: (12575, 11)


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


## 2. Genel Veri Sağlığı Kontrolü

Temizliğe başlamadan önce verinin genel durumunu anlamak gerekir:

- `df.info()` → veri tipleri ve boş değer sayıları
- `df.isnull().sum()` → sütun bazında eksik değer sayısı
- `df.duplicated().sum()` → tekrar eden satır sayısı

In [2]:
df_ham.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(8)
memory usage: 1.1+ MB


In [3]:
print("Sütun bazında eksik değer sayısı:")
print(df_ham.isnull().sum())

print("\nToplam tekrar eden satır sayısı:", df_ham.duplicated().sum())

Sütun bazında eksik değer sayısı:
Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64

Toplam tekrar eden satır sayısı: 0


## 3. Eksik Değerlerin Tespiti ve Giderilmesi


In [4]:
# Item bilgisi eksik olan satırları inceleyelim
eksik_item = df_ham[df_ham['Item'].isnull()]

print(f"Item bilgisi eksik olan satır sayısı: {len(eksik_item)}")

eksik_item.head()

Item bilgisi eksik olan satır sayısı: 1213


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
5,TXN_7482416,CUST_09,Patisserie,NaN,NaN,10.0,200.0,Credit Card,Online,2023-11-30,NaN
7,TXN_1372952,CUST_21,Furniture,NaN,33.5,NaN,NaN,Digital Wallet,In-store,2024-04-02,True
11,TXN_5422631,CUST_09,Milk Products,NaN,NaN,8.0,52.0,Digital Wallet,In-store,2025-01-12,True
15,TXN_1809665,CUST_14,Beverages,NaN,24.5,NaN,NaN,Credit Card,In-store,2022-05-11,NaN
17,TXN_9634894,CUST_15,Milk Products,NaN,NaN,10.0,275.0,Digital Wallet,Online,2022-04-17,NaN


In [5]:
df_temiz = df_ham.copy()

In [6]:
# Eksik Item değerlerini "Bilinmiyor" ile dolduruyoruz
df_temiz['Item'] = df_temiz['Item'].fillna('Bilinmiyor')

print(
    "Doldurma sonrası eksik değer sayısı:",
    df_temiz['Item'].isnull().sum()
)

df_temiz['Item'].value_counts()

Doldurma sonrası eksik değer sayısı: 0


,count
Item,
Bilinmiyor,1213
Item_2_BEV,126
Item_25_FUR,113
Item_11_FUR,110
Item_16_MILK,109
...,...
Item_5_BEV,7
Item_13_BEV,7
Item_13_FUR,7


In [7]:
# Anlamlı bir alt kümeye göre (müşteri + tarih + ürün) tekrar kontrolü

tekrar_sayisi = df_temiz.duplicated(
    subset=['Customer ID', 'Transaction Date', 'Item']
).sum()

print(f"Müşteri+tarih+ürün bazında tekrar eden kayıt sayısı: {tekrar_sayisi}")

Müşteri+tarih+ürün bazında tekrar eden kayıt sayısı: 40


## 5. Veri Tiplerinin Düzeltilmesi

In [8]:
print("Dönüşümden önce tip:", df_temiz['Transaction Date'].dtype)
print("Örnek değer:", df_temiz['Transaction Date'].iloc[0])

df_temiz['Transaction Date'] = pd.to_datetime(
    df_temiz['Transaction Date'],
    errors='coerce'
)

print("\nDönüşümden sonra tip:", df_temiz['Transaction Date'].dtype)
print("Çevrilemeyen (NaT) satır sayısı:", df_temiz['Transaction Date'].isnull().sum())

# Artık tarihten yıl/ay bilgilerini çıkarabiliriz
df_temiz['islem_yili'] = df_temiz['Transaction Date'].dt.year
df_temiz['islem_ayi'] = df_temiz['Transaction Date'].dt.month

df_temiz[['Transaction Date', 'islem_yili', 'islem_ayi']].head()

Dönüşümden önce tip: object
Örnek değer: 2024-04-08

Dönüşümden sonra tip: datetime64[ns]
Çevrilemeyen (NaT) satır sayısı: 0


,Transaction Date,islem_yili,islem_ayi
0,2024-04-08,2024,4
1,2023-07-23,2023,7
2,2022-10-05,2022,10
3,2022-05-07,2022,5
4,2022-10-02,2022,10


## 6. Mantıksal Hata / Aykırı Değer Tespiti ve Düzeltilmesi

In [9]:
mantiksiz = df_temiz[
    (df_temiz['Quantity'] <= 0) |
    (df_temiz['Price Per Unit'] <= 0)
]

print(f"Mantıksız miktar/fiyat içeren satır sayısı: {len(mantiksiz)}")

mantiksiz[['Item', 'Quantity', 'Price Per Unit', 'Total Spent']]

Mantıksız miktar/fiyat içeren satır sayısı: 0


,Item,Quantity,Price Per Unit,Total Spent


##Mantıksız Değerlerin Düzeltilmesi
Veri setinde yapılan kontroller sonucunda Quantity ve Price Per Unit sütunlarında negatif veya mantıksız bir değere rastlanmamıştır. Ancak veri temizleme sürecinde uygulanabilecek düzeltme yöntemini göstermek amacıyla, olası negatif değerleri pozitif hale getiren ve ardından Total Spent değerini yeniden hesaplayan örnek bir düzeltme işlemi uygulanmıştır.

In [10]:
# Düzeltme: negatif Quantity ve Price Per Unit değerlerini pozitif yap

df_temiz['Quantity'] = df_temiz['Quantity'].abs()
df_temiz['Price Per Unit'] = df_temiz['Price Per Unit'].abs()

# Total Spent değerini yeniden hesapla
df_temiz['Total Spent'] = (
    df_temiz['Quantity'] * df_temiz['Price Per Unit']
)

# Kontrol: negatif değer kaldı mı?
print(
    "Düzeltme sonrası negatif Quantity sayısı:",
    (df_temiz['Quantity'] < 0).sum()
)

print(
    "Düzeltme sonrası negatif Price Per Unit sayısı:",
    (df_temiz['Price Per Unit'] < 0).sum()
)

Düzeltme sonrası negatif Quantity sayısı: 0
Düzeltme sonrası negatif Price Per Unit sayısı: 0


## 7. Kategori Tutarlılığı Kontrolü

In [11]:
# Kategorik sütunlardaki benzersiz değerleri kontrol edelim

print("Kategoriler:")
print(sorted(df_temiz['Category'].dropna().unique()))

print("\nÖdeme Yöntemleri:")
print(sorted(df_temiz['Payment Method'].dropna().unique()))

print("\nLokasyonlar:")
print(sorted(df_temiz['Location'].dropna().unique()))

Kategoriler:
['Beverages', 'Butchers', 'Computers and electric accessories', 'Electric household essentials', 'Food', 'Furniture', 'Milk Products', 'Patisserie']

Ödeme Yöntemleri:
['Cash', 'Credit Card', 'Digital Wallet']

Lokasyonlar:
['In-store', 'Online']


In [12]:
# Baştaki/sondaki boşluk kontrolü

kategori_bosluk = df_temiz['Category'].dropna().apply(
    lambda x: x != x.strip()
).sum()

odeme_bosluk = df_temiz['Payment Method'].dropna().apply(
    lambda x: x != x.strip()
).sum()

lokasyon_bosluk = df_temiz['Location'].dropna().apply(
    lambda x: x != x.strip()
).sum()

print("Başta/sonda boşluk içeren Category sayısı:", kategori_bosluk)
print("Başta/sonda boşluk içeren Payment Method sayısı:", odeme_bosluk)
print("Başta/sonda boşluk içeren Location sayısı:", lokasyon_bosluk)

Başta/sonda boşluk içeren Category sayısı: 0
Başta/sonda boşluk içeren Payment Method sayısı: 0
Başta/sonda boşluk içeren Location sayısı: 0


In [13]:
# Kategorik değerleri standartlaştırma

df_temiz['Category'] = df_temiz['Category'].str.strip().str.title()
df_temiz['Payment Method'] = df_temiz['Payment Method'].str.strip().str.title()
df_temiz['Location'] = df_temiz['Location'].str.strip().str.title()

print("Kategori tutarlılığı kontrolü ve standartlaştırma tamamlandı.")

Kategori tutarlılığı kontrolü ve standartlaştırma tamamlandı.


## 8. İndeks Düzenleme

In [14]:
df_temiz = df_temiz.reset_index(drop=True)
print("İndeks sıfırlandı. İlk 5 satır:")
df_temiz.head()

İndeks sıfırlandı. İlk 5 satır:


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied,islem_yili,islem_ayi
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True,2024,4
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True,2023,7
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False,2022,10
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN,2022,5
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False,2022,10


## 9. Temizlenmiş Veriyi Kaydetme

In [15]:
df_temiz.to_csv('/content/retail_store_sales_temiz.csv', index=False)

print("Temizlenmiş veri 'retail_store_sales_temiz.csv' olarak kaydedildi.")

print("\nSon durumun genel bilgisi:")
df_temiz.info()

Temizlenmiş veri 'retail_store_sales_temiz.csv' olarak kaydedildi.

Son durumun genel bilgisi:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    12575 non-null  object        
 1   Customer ID       12575 non-null  object        
 2   Category          12575 non-null  object        
 3   Item              12575 non-null  object        
 4   Price Per Unit    11966 non-null  float64       
 5   Quantity          11971 non-null  float64       
 6   Total Spent       11362 non-null  float64       
 7   Payment Method    12575 non-null  object        
 8   Location          12575 non-null  object        
 9   Transaction Date  12575 non-null  datetime64[ns]
 10  Discount Applied  8376 non-null   object        
 11  islem_yili        12575 non-null  int32         
 12  islem_ayi         12575 non-null  i

## 10. Uçtan Uca Temizleme Fonksiyonu

In [16]:
def veriyi_temizle(df_ham):
    """
    Ham retail store sales DataFrame'ini alır,
    temel temizleme adımlarını uygular
    ve temizlenmiş DataFrame'i döndürür.
    """

    df = df_ham.copy()

    # 1) Eksik Item değerlerini doldur
    df['Item'] = df['Item'].fillna('Bilinmiyor')

    # 2) Tarih sütununu datetime'a çevir
    df['Transaction Date'] = pd.to_datetime(
        df['Transaction Date'],
        errors='coerce'
    )

    # 3) Negatif Quantity ve Price Per Unit değerlerini düzelt
    df['Quantity'] = df['Quantity'].abs()
    df['Price Per Unit'] = df['Price Per Unit'].abs()

    # Total Spent değerini yeniden hesapla
    df['Total Spent'] = (
        df['Quantity'] * df['Price Per Unit']
    )

    # 4) Tekrar eden kayıtları temizle
    df = df.drop_duplicates(
        subset=['Customer ID', 'Transaction Date', 'Item'],
        keep='first'
    )

    # 5) Kategorik sütunları standartlaştır
    df['Category'] = df['Category'].str.strip().str.title()
    df['Payment Method'] = df['Payment Method'].str.strip().str.title()
    df['Location'] = df['Location'].str.strip().str.title()

    # 6) İndeksi sıfırla
    df = df.reset_index(drop=True)

    return df


# Fonksiyonu ham veri üzerinde test edelim
df_sonuc = veriyi_temizle(df_ham)

print("Temizleme sonrası kontrol:")
print("- Eksik Item:", df_sonuc['Item'].isnull().sum())
print("- Negatif Quantity:", (df_sonuc['Quantity'] < 0).sum())
print("- Negatif Price Per Unit:", (df_sonuc['Price Per Unit'] < 0).sum())
print("- Transaction Date tipi:", df_sonuc['Transaction Date'].dtype)

df_sonuc.head()

Temizleme sonrası kontrol:
- Eksik Item: 0
- Negatif Quantity: 0
- Negatif Price Per Unit: 0
- Transaction Date tipi: datetime64[ns]


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False
